# Phase 01 — Step 14
## Evidence-Grounded OpenAI Recommendation System

This notebook applies the Phase 02 recommendation design to Internship Experience → Employability.

The frozen Phase 01 XGBoost model produces the employability prediction. OpenAI does **not** calculate or change that score. It only converts validated model evidence into personalized guidance.

Evidence used:

- construct scores (`Clear_Objectives`, `School_Support`, `Workplace_Environment`, `Career_Development`)
- item-level SIE scores
- SHAP contributions from the frozen XGBoost model
- a verified learning-resource library
- a rule-based fallback if the OpenAI API is unavailable

The OpenAI key is loaded from `samindi/Employability/.env`.

Phase 02 and Phase 03 notebooks, models, and services are not modified.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from employability_service import build_phase_services

services = build_phase_services(PROJECT_DIR)
phase1 = services["phase1"]
print(phase1.health())

In [ ]:
sample_responses = {
    "SIE1": 4, "SIE2": 5, "SIE3": 4, "SIE4": 4,
    "SIE5": 3, "SIE6": 3,
    "SIE7": 5, "SIE8": 4, "SIE9": 4, "SIE10": 3,
    "SIE11": 3, "SIE12": 3,
}

result = phase1.predict(sample_responses)

print("Predicted employability:", result["prediction"]["evs_avg"])
print("Recommendation source:", result.get("recommendation_source"))
print("Construct scores:", result["construct_scores"])

In [ ]:
guidance = result.get("ai_recommendation", {})

print(guidance.get("overall_career_readiness_summary", ""))
print("\nTop strengths:")
for item in guidance.get("top_3_strengths", []):
    print("-", item)

print("\nPriority gaps:")
for item in guidance.get("top_3_areas_for_improvement", []):
    print("-", item)

In [ ]:
OUTPUT_DIR = PROJECT_DIR / "reports" / "phase1" / "recommendation_system"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame([
    {
        "Predicted_Employability": result["prediction"]["evs_avg"],
        "Recommendation_Source": result.get("recommendation_source"),
        "Clear_Objectives": result["construct_scores"]["Clear_Objectives"],
        "School_Support": result["construct_scores"]["School_Support"],
        "Workplace_Environment": result["construct_scores"]["Workplace_Environment"],
        "Career_Development": result["construct_scores"]["Career_Development"],
    }
]).to_csv(OUTPUT_DIR / "sample_recommendation_evidence.csv", index=False)

import json
(OUTPUT_DIR / "sample_recommendation_output.json").write_text(
    json.dumps(result, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved Phase 01 recommendation evidence to", OUTPUT_DIR)

## Research note

The language model is a presentation layer only. The frozen Phase 01 XGBoost prediction is unchanged. If `OPENAI_API_KEY` is missing, the same evidence is converted by the rule-based fallback used in Phase 02.